In [2]:
import pandas as pd
import numpy as np
import random
import uuid
from faker import Faker
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import joblib


In [3]:
fake = Faker()

locations = [
    "Bangalore", "Mumbai", "Delhi", "Chennai",
    "Hyderabad", "Pune", "Jaipur", "Goa",
    "Kolkata", "Ahmedabad"
]

amenities_master = [
    "WiFi", "Parking", "Kitchen",
    "Pool", "Gym", "Fireplace", "Security"
]

data = []

for i in range(1000):
    location = random.choice(locations)
    
    guests = random.randint(1, 10)
    bedrooms = random.randint(1, 5)
    beds = random.randint(1, 6)
    bathrooms = random.randint(1, 4)
    
    amenities = random.sample(
        amenities_master,
        random.randint(2, 6)
    )
    
    base_price = {
        "Bangalore": 2500,
        "Mumbai": 4000,
        "Delhi": 3500,
        "Chennai": 2800,
        "Hyderabad": 2600,
        "Pune": 2400,
        "Jaipur": 2000,
        "Goa": 4500,
        "Kolkata": 2200,
        "Ahmedabad": 2100
    }
    
    price = base_price[location] + random.randint(-800, 2000)
    
    data.append({
        "_id": str(uuid.uuid4()),
        "title": fake.company() + " Stay",
        "location": location,
        "price": price,
        "guests": guests,
        "bedrooms": bedrooms,
        "beds": beds,
        "bathrooms": bathrooms,
        "amenities": amenities
    })

df = pd.DataFrame(data)

df.head()

,_id,title,location,price,guests,bedrooms,beds,bathrooms,amenities
0,9b7cad05-d7f0-4259-b97b-30d8d9020121,Johnson-Richmond Stay,Delhi,5450,10,3,5,2,"[Fireplace, Security, Parking, Kitchen, Gym]"
1,71ee337c-18a5-4942-97ad-94771cd0e251,Foster-Herman Stay,Ahmedabad,3959,4,5,4,4,"[Pool, Kitchen, WiFi]"
2,c9f4b38d-610a-4de5-8340-1225e47f5777,Hicks-Fry Stay,Mumbai,3763,10,5,4,3,"[Parking, Fireplace]"
3,a89e70dd-0ca5-4909-a3a2-ddb903968b82,"Rodriguez, Wilson and Rogers Stay",Jaipur,3849,8,2,6,2,"[Gym, WiFi]"
4,bcf29f40-9203-4ec9-bc80-143064acbce5,Ayers-Schwartz Stay,Chennai,2485,7,1,4,4,"[Security, Gym, Parking, Fireplace, Kitchen, W..."


In [4]:
amenities_list = [
    "WiFi", "Parking", "Kitchen",
    "Pool", "Gym", "Fireplace", "Security"
]

for amenity in amenities_list:
    df[amenity.lower()] = df["amenities"].apply(
        lambda x: 1 if amenity in x else 0
    )

In [6]:
# Convert ObjectId to string
df["_id"] = df["_id"].astype(str)

# Extract nested details
df["guests"] = df["details"].apply(lambda x: x.get("guests", 0))
df["bedrooms"] = df["details"].apply(lambda x: x.get("bedrooms", 0))
df["beds"] = df["details"].apply(lambda x: x.get("beds", 0))
df["bathrooms"] = df["details"].apply(lambda x: x.get("bathrooms", 0))


In [7]:
amenities_list = [
    "WiFi", "Parking", "Kitchen",
    "Pool", "Gym", "Fireplace", "Security"
]

for amenity in amenities_list:
    df[amenity.lower()] = df["amenities"].apply(
        lambda x: 1 if amenity in x else 0
    )

df.head()


,_id,title,description,location,price,image,amenities,details,hostId,createdAt,...,bedrooms,beds,bathrooms,wifi,parking,kitchen,pool,gym,fireplace,security
0,6852701a8de5ff0a7428e896,Luxury Villa with Ocean View,Experience the magic of Santorini in this stun...,Bangalore,447,https://images.pexels.com/photos/1396122/pexel...,"[Pool, Fireplace, Kitchen, WiFi]","{'guests': 30, 'bedrooms': 12, 'beds': 30, 'ba...",685076754f7378630c99e82b,2025-06-18 07:51:54.561,...,12,30,2,1,0,1,1,0,1,0
1,6852708b8de5ff0a7428e897,Modern Apartment in City Center,Experience the magic of Santorini in this stun...,Bangalore,180,https://images.pexels.com/photos/1571460/pexel...,"[WiFi, Gym, Parking, Security]","{'guests': 40, 'bedrooms': 3, 'beds': 31, 'bat...",685076754f7378630c99e82b,2025-06-18 07:53:47.585,...,3,31,1,1,1,0,0,1,0,1
2,68528a758de5ff0a7428e898,Cozy Cabin in the Mountains,Experience the magic of Santorini in this stun...,Bangalore,318,https://images.pexels.com/photos/338504/pexels...,"[Security, Fireplace, Kitchen, Gym, WiFi]","{'guests': 37, 'bedrooms': 3, 'beds': 4, 'bath...",685076754f7378630c99e82b,2025-06-18 09:44:21.527,...,3,4,1,1,0,1,0,1,1,1
3,68528ad58de5ff0a7428e899,Beachfront Bungalow,Experience the magic of Santorini in this stun...,Bangalore,889,https://images.pexels.com/photos/189296/pexels...,"[Security, Gym, Pool]","{'guests': 9, 'bedrooms': 1, 'beds': 9, 'bathr...",685076754f7378630c99e82b,2025-06-18 09:45:57.010,...,1,9,2,0,0,0,1,1,0,1
4,699564acd222d386dab5b942,Royal Palm Residency,"Luxury hotel offering spacious AC rooms, free ...",Mumbai,10368,https://images.pexels.com/photos/189296/pexels...,"[WiFi, Kitchen, Gym]","{'guests': 145, 'bedrooms': 1, 'beds': 1, 'bat...",685191fb1284e8db62998cb6,2026-02-18 07:05:16.193,...,1,1,1,1,0,1,0,1,0,0


In [5]:
le_location = LabelEncoder()
df["location_encoded"] = le_location.fit_transform(df["location"])

In [6]:
# Location weight boost
df["location_encoded"] = df["location_encoded"] * 3

# Scale price separately
price_scaler = StandardScaler()
df["price_scaled"] = price_scaler.fit_transform(df[["price"]])

In [7]:
feature_cols = [
    "location_encoded",
    "guests",
    "bedrooms",
    "beds",
    "bathrooms",
    "wifi",
    "parking",
    "kitchen",
    "pool",
    "gym",
    "fireplace",
    "security"
]

features = df[feature_cols]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [8]:
# Feature similarity
feature_similarity = cosine_similarity(scaled_features)

# Price similarity
price_similarity = cosine_similarity(
    df[["price_scaled"]]
)

# Hybrid weighted similarity
hybrid_similarity = (
    0.7 * feature_similarity +
    0.3 * price_similarity
)

In [9]:
def recommend_property_v2(listing_id, top_n=5):
    
    if listing_id not in df["_id"].values:
        print("Listing not found")
        return
    
    index = df[df["_id"] == listing_id].index[0]
    current_location = df.iloc[index]["location"]
    
    similarity_scores = list(enumerate(hybrid_similarity[index]))
    
    # Location priority boost
    boosted_scores = []
    
    for i, score in similarity_scores:
        if df.iloc[i]["location"] == current_location:
            score += 0.15
        boosted_scores.append((i, score))
    
    boosted_scores = sorted(
        boosted_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    top_indices = [i[0] for i in boosted_scores[1:top_n+1]]
    
    return df.iloc[top_indices][[
        "_id", "title", "location", "price"
    ]]

In [13]:
test_id = df["_id"].iloc[0]
recommend_property(test_id)


,_id,title,location,price,image
7,69969720ab2249b1ace3f5d3,The Royal Heritage Palace,Hyderabad,16331,https://images.pexels.com/photos/161758/govern...
9,699697e9ab2249b1ace3f5d5,Green Leaf Eco Hotel,Chennai,14472,https://images.pexels.com/photos/7154962/pexel...
6,69957760d222d386dab5b944,Ocean View Grand,Mumbai,13185,https://images.pexels.com/photos/5029795/pexel...
3,68528ad58de5ff0a7428e899,Beachfront Bungalow,Bangalore,889,https://images.pexels.com/photos/189296/pexels...
1,6852708b8de5ff0a7428e897,Modern Apartment in City Center,Bangalore,180,https://images.pexels.com/photos/1571460/pexel...


In [10]:
test_id = df["_id"].iloc[0]

recommend_property_v2(test_id)

,_id,title,location,price
195,daea5524-5e5c-4021-9a7a-1414870c7bb0,Jones Ltd Stay,Delhi,3916
903,f23a9e79-f420-4651-bb85-2d06c0f2f499,"Sosa, Wilson and Singleton Stay",Delhi,4702
198,4bddbe8e-f0d4-4025-aba4-743990a4581c,"Manning, Blanchard and Rice Stay",Delhi,4474
763,c93e4517-93eb-4ccd-a8e2-11ede88cf42c,Gilbert-Navarro Stay,Delhi,3843
788,493e6ba8-89bd-4a2b-aff5-53da23811902,Morgan-Dickson Stay,Delhi,5475


In [11]:
joblib.dump(hybrid_similarity, "similarity_v2.pkl")
joblib.dump(df, "recommend_data_v2.pkl")

print("🔥 StayFinder V2 model saved successfully!")

🔥 StayFinder V2 model saved successfully!
